# ORI 2026 - Predviđanje cenovnog trenda - VALUTE (Podtim 5)

Ovaj notebook radi sve faze u nizu:
1. **Faza 1** - Priprema podataka (Currency Data ALL + missing values)
2. **Faza 2+3** - Inženjering ulaza (indikatori) i izlaza
3. **Faza 5+6** - Model neuronske mreže (MLP) i eksperimenti

**Pre pokretanja:**
- Stavi CSV fajlove (`EURUSD_Data.csv`, `USDCAD_Data.csv`, ...) u isti folder kao ovaj notebook
- Instaliraj pakete: `pip install pandas numpy openpyxl scikit-learn` (u terminalu, u istom environment-u koji koristi Jupyter)
- Pokreni ćelije od vrha do dna (Shift+Enter)


In [14]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
from math import sqrt

warnings.filterwarnings("ignore")

# ── KONFIGURACIJA ──────────────────────────────────────────────────────
INPUT_DIR  = ""  # folder gde su *_Data.csv fajlovi (default: ovaj folder)
ID_COLUMN  = "Pair"

CURRENCY_ALL_FILE     = "../asd/Currency Data ALL.xlsx"
CURRENCY_FEATURES_FILE = "../asd/Currency Data Features.xlsx"
NN_RESULTS_FILE       = "../asd/NN_Experiment_Results.xlsx"

print("Trenutni radni folder:", os.getcwd())
print("Fajlovi u folderu:")
for f in os.listdir(""):
    print("  -", f)


Trenutni radni folder: /Users/djordjeradosavljevic/Downloads/fax/N/Projekat/v_p
Fajlovi u folderu:
  - USDJPY Data.csv
  - rez
  - .DS_Store
  - EURUSD Data.csv
  - NN_Predikcija_Vizuelizacija.ipynb
  - grafovi
  - NN_Experiment_Results.xlsx
  - Currency Data ALL.xlsx
  - README.md
  - Currency Data Features.xlsx
  - NN_Praktični_Primeri.ipynb
  - USDCHF Data.csv
  - USDCAD Data.csv
  - .venv
  - ORI_2026_Valute.ipynb
  - ORI_2026_Grafikoni.ipynb
  - USDGBP Data.csv
  - faza2_3_indikatori.py
  - valute_faza1.py
  - faza5_6_eksperimenti.py
  - .idea


## Faza 1: Priprema podataka - Currency Data ALL

In [2]:
def load_currency_csv(path: str, pair_name: str) -> pd.DataFrame:
    """Učitava CSV, izvlači Date i Price, vraća long format: Pair | Date | Price"""
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip().strip('"') for c in df.columns]

    df = df[["Date", "Price"]].copy()
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
    df["Price"] = pd.to_numeric(df["Price"].astype(str).str.replace(",", ""), errors="coerce")
    df["Pair"] = pair_name
    return df[["Pair", "Date", "Price"]]


def build_currency_data_all(input_dir: str) -> pd.DataFrame:
    csv_files = sorted(glob.glob(os.path.join(input_dir, "*_Data.csv")))
    if not csv_files:
        raise FileNotFoundError(
            f"Nije pronađen nijedan *_Data.csv fajl u folderu: {input_dir}\n"
            f"Provери INPUT_DIR ili nazive fajlova."
        )

    print(f"Pronađeno fajlova: {len(csv_files)}")
    frames = []
    for path in csv_files:
        fname = os.path.basename(path)
        pair_name = fname.replace("_Data.csv", "")
        df = load_currency_csv(path, pair_name)
        print(f"  {pair_name}: {len(df)} redova, od {df['Date'].min().date()} do {df['Date'].max().date()}")
        frames.append(df)

    result = pd.concat(frames, ignore_index=True)
    result = result.sort_values(["Pair", "Date"]).reset_index(drop=True)
    print(f"\nUkupno: {result.shape[0]} redova, {result['Pair'].nunique()} parova")
    return result


def analyze_missing_values(currency_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for pair, g in currency_all.groupby("Pair"):
        total_rows = len(g)
        missing = g["Price"].isna().sum()
        pct = round(missing / total_rows * 100, 4) if total_rows else 0
        rows.append({
            "Pair": pair,
            "total_rows": total_rows,
            "missing_price_count": missing,
            "missing_price_pct": pct,
            "date_min": g["Date"].min(),
            "date_max": g["Date"].max(),
        })
    return pd.DataFrame(rows).sort_values("Pair").reset_index(drop=True)


In [4]:
def build_currency_data_all(input_dir: str) -> pd.DataFrame:
    csv_files = sorted(glob.glob(os.path.join(input_dir, "* Data.csv")))
    if not csv_files:
        raise FileNotFoundError(f"Nije pronađen nijedan '* Data.csv' fajl u folderu: {input_dir}")

    print(f"Pronađeno fajlova: {len(csv_files)}")
    frames = []
    for path in csv_files:
        fname = os.path.basename(path)
        pair_name = fname.replace(" Data.csv", "")
        df = load_currency_csv(path, pair_name)
        print(f"  {pair_name}: {len(df)} redova, od {df['Date'].min().date()} do {df['Date'].max().date()}")
        frames.append(df)

    result = pd.concat(frames, ignore_index=True)
    result = result.sort_values(["Pair", "Date"]).reset_index(drop=True)
    print(f"\nUkupno: {result.shape[0]} redova, {result['Pair'].nunique()} parova")
    return result


currency_all = build_currency_data_all(INPUT_DIR)
currency_all.head()

,Pair,Date,Price
0,EURUSD,2016-01-01,1.0858
1,EURUSD,2016-01-04,1.0829
2,EURUSD,2016-01-05,1.0746
3,EURUSD,2016-01-06,1.0778
4,EURUSD,2016-01-07,1.0934


In [5]:
currency_all = build_currency_data_all(INPUT_DIR)
currency_all.head()


,Pair,Date,Price
0,EURUSD,2016-01-01,1.0858
1,EURUSD,2016-01-04,1.0829
2,EURUSD,2016-01-05,1.0746
3,EURUSD,2016-01-06,1.0778
4,EURUSD,2016-01-07,1.0934


In [6]:
missing_summary = analyze_missing_values(currency_all)
missing_summary


,Pair,total_rows,missing_price_count,missing_price_pct,date_min,date_max
0,EURUSD,2610,0,0.0,2016-01-01,2026-01-01
1,USDCAD,2610,0,0.0,2016-01-01,2026-01-01
2,USDCHF,2610,0,0.0,2016-01-01,2026-01-01
3,USDGBP,2610,0,0.0,2016-01-01,2026-01-01
4,USDJPY,2610,0,0.0,2016-01-01,2026-01-01


In [15]:
# import os
# print("Working directory:", os.getcwd())
# print("Files here:", os.listdir("."))

Working directory: /Users/djordjeradosavljevic/Downloads/fax/N/Projekat/v_p
Files here: ['USDJPY Data.csv', 'EURUSD Data.csv', 'Currency Data ALL.xlsx', 'USDCHF Data.csv', 'USDCAD Data.csv', '.venv', 'ORI_2026_Valute.ipynb', 'USDGBP Data.csv', 'faza2_3_indikatori.py', 'valute_faza1.py', 'faza5_6_eksperimenti.py', '.idea']


In [7]:
# Snimi rezultate Faze 1

with pd.ExcelWriter(CURRENCY_ALL_FILE, engine="openpyxl", datetime_format="YYYY-MM-DD") as writer:
    currency_all.to_excel(writer, sheet_name="Currency_Data_ALL", index=False)
    missing_summary.to_excel(writer, sheet_name="Missing_Summary", index=False)
print(f"Sačuvano: {CURRENCY_ALL_FILE}")


Sačuvano: Currency Data ALL.xlsx


## Faza 2+3: Inženjering ulaza (indikatori) i izlaza

Računa 13 baznih indikatora (SMA, EMA, KAMA, MACD x3, ROC, SAR, CCI, RSI, CMO, RCI, STD),
SMA od indikatora 5-11, ROC od svih indikatora 1-11, i izlazne promenljive
(Price, Price_ROC, Price_SMA, Price_SMA_ROC).


In [8]:
PARAMS = {
    "SMA_period"  : 14,
    "EMA_period"  : 14,
    "KAMA_period" : 14, #na 14
    "MACD_fast"   : 12,
    "MACD_slow"   : 26,
    "MACD_signal" : 9,
    "ROC_period"  : 14,  #14
    "SAR_af"      : 0.025,
    "SAR_max_af"  : 0.25,
    "CCI_period"  : 14,
    "RSI_period"  : 14,
    "CMO_period"  : 14,
    "RCI_period"  : 14,
    "STD_period"  : 14,
    "SMA_of_ind"  : 9,
    "ROC_of_ind"  : 9,
    "SMA_output"  : 9,
    "ROC_output"  : 9,
}


def _sma(series, period):
    return series.rolling(window=period, min_periods=1).mean()

def _ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

def _kama(series, period=10, fast=2, slow=30):
    n = len(series)
    kama_vals = np.full(n, np.nan)
    sc_fast = 2 / (fast + 1)
    sc_slow = 2 / (slow + 1)
    prices = series.values

    first_valid = 0
    while first_valid < n and np.isnan(prices[first_valid]):
        first_valid += 1
    if first_valid >= n:
        return pd.Series(kama_vals, index=series.index)

    kama_vals[first_valid] = prices[first_valid]
    for i in range(first_valid + 1, n):
        if i < period:
            kama_vals[i] = prices[i]
            continue
        direction = abs(prices[i] - prices[i - period])
        volatility = np.nansum(np.abs(np.diff(prices[i - period: i + 1])))
        er = direction / volatility if volatility != 0 else 0
        sc = (er * (sc_fast - sc_slow) + sc_slow) ** 2
        kama_vals[i] = kama_vals[i - 1] + sc * (prices[i] - kama_vals[i - 1])
    return pd.Series(kama_vals, index=series.index)

def _macd(series, fast, slow, signal):
    ema_fast = _ema(series, fast)
    ema_slow = _ema(series, slow)
    macd_line = ema_fast - ema_slow
    signal_line = _ema(macd_line, signal)
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

def _roc(series, period):
    return series.pct_change(periods=period) * 100

def _sar(high, low, af_start=0.02, af_max=0.2):
    sar = np.full(len(high), np.nan)
    trend = 1
    sar[0] = low.iloc[0]
    ep = high.iloc[0]
    af = af_start

    for i in range(1, len(high)):
        prev_sar = sar[i - 1]
        if trend == 1:
            sar[i] = prev_sar + af * (ep - prev_sar)
            sar[i] = min(sar[i], low.iloc[i - 1], low.iloc[i - 2] if i >= 2 else low.iloc[i - 1])
            if low.iloc[i] < sar[i]:
                trend = -1
                sar[i] = ep
                ep = low.iloc[i]
                af = af_start
            else:
                if high.iloc[i] > ep:
                    ep = high.iloc[i]
                    af = min(af + af_start, af_max)
        else:
            sar[i] = prev_sar - af * (prev_sar - ep)
            sar[i] = max(sar[i], high.iloc[i - 1], high.iloc[i - 2] if i >= 2 else high.iloc[i - 1])
            if high.iloc[i] > sar[i]:
                trend = 1
                sar[i] = ep
                ep = high.iloc[i]
                af = af_start
            else:
                if low.iloc[i] < ep:
                    ep = low.iloc[i]
                    af = min(af + af_start, af_max)
    return pd.Series(sar, index=high.index)

def _cci(high, low, close, period):
    typical = (high + low + close) / 3
    sma_tp = typical.rolling(period, min_periods=1).mean()
    mad = typical.rolling(period, min_periods=1).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    return (typical - sma_tp) / (0.015 * mad)

def _rsi(series, period):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period, min_periods=1).mean()
    loss = (-delta.clip(upper=0)).rolling(period, min_periods=1).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def _cmo(series, period):
    delta = series.diff()
    up = delta.clip(lower=0).rolling(period, min_periods=1).sum()
    down = (-delta.clip(upper=0)).rolling(period, min_periods=1).sum()
    total = up + down
    return ((up - down) / total.replace(0, np.nan)) * 100

def _rci(series, period):
    def _rci_single(window):
        n = len(window)
        price_rank = pd.Series(window).rank(ascending=False).values
        time_rank = np.arange(1, n + 1)
        d2 = np.sum((price_rank - time_rank) ** 2)
        return (1 - 6 * d2 / (n * (n**2 - 1))) * 100
    return series.rolling(period, min_periods=period).apply(_rci_single, raw=True)

def _std(series, period):
    return series.rolling(period, min_periods=1).std()


print("Funkcije za indikatore definisane.")


Funkcije za indikatore definisane.


In [9]:
def compute_indicators(price_series: pd.Series, p: dict) -> pd.DataFrame:
    close = price_series.copy().astype(float)
    high = close
    low = close

    cols = {}
    cols["SMA"] = _sma(close, p["SMA_period"])
    cols["EMA"] = _ema(close, p["EMA_period"])
    cols["KAMA"] = _kama(close, p["KAMA_period"])

    macd_line, macd_signal, macd_hist = _macd(close, p["MACD_fast"], p["MACD_slow"], p["MACD_signal"])
    cols["MACD"] = macd_line
    cols["MACD_signal"] = macd_signal
    cols["MACD_hist"] = macd_hist

    ind5_11 = {}
    ind5_11["ROC"] = _roc(close, p["ROC_period"])
    ind5_11["SAR"] = _sar(high, low, p["SAR_af"], p["SAR_max_af"])
    ind5_11["CCI"] = _cci(high, low, close, p["CCI_period"])
    ind5_11["RSI"] = _rsi(close, p["RSI_period"])
    ind5_11["CMO"] = _cmo(close, p["CMO_period"])
    ind5_11["RCI"] = _rci(close, p["RCI_period"])
    ind5_11["STD"] = _std(close, p["STD_period"])

    for name, series in ind5_11.items():
        cols[name] = series
        cols[f"{name}_SMA"] = _sma(series, p["SMA_of_ind"])

    all_ind_names = ["SMA", "EMA", "KAMA", "MACD", "MACD_signal", "MACD_hist"] + list(ind5_11.keys())
    for name in all_ind_names:
        cols[f"{name}_ROC"] = _roc(cols[name], p["ROC_of_ind"])

    return pd.DataFrame(cols, index=close.index)


def compute_outputs(price_series: pd.Series, p: dict) -> pd.DataFrame:
    close = price_series.copy().astype(float)
    price_roc = _roc(close, p["ROC_output"])
    price_sma = _sma(close, p["SMA_output"])
    return pd.DataFrame({
        "Price": close,
        "Price_ROC": price_roc,
        "Price_SMA": price_sma,
        "Price_SMA_ROC": _sma(price_roc, p["SMA_output"]),
    }, index=close.index)


def process_all_series(data_all: pd.DataFrame, id_column: str, params: dict) -> pd.DataFrame:
    ids = data_all[id_column].unique()
    print(f"Pronađeno {len(ids)} serija u koloni '{id_column}': {list(ids)}")

    all_frames = []
    for item_id in ids:
        sub = data_all[data_all[id_column] == item_id].copy()
        sub = sub.sort_values("Date").dropna(subset=["Price"]).reset_index(drop=True)

        if len(sub) < 30:
            print(f"  {item_id}: premalo podataka ({len(sub)} redova), preskačem.")
            continue

        price_series = sub.set_index("Date")["Price"]
        indicators = compute_indicators(price_series, params)
        outputs = compute_outputs(price_series, params)

        combined = indicators.join(outputs).reset_index()
        combined.insert(0, id_column, item_id)

        all_frames.append(combined)
        print(f"  {item_id}: {len(combined)} redova, {combined.shape[1]} kolona")

    return pd.concat(all_frames, ignore_index=True)


In [10]:
features = process_all_series(currency_all, ID_COLUMN, PARAMS)
features.head()


,Pair,Date,SMA,EMA,KAMA,MACD,MACD_signal,MACD_hist,ROC,ROC_SMA,...,SAR_ROC,CCI_ROC,RSI_ROC,CMO_ROC,RCI_ROC,STD_ROC,Price,Price_ROC,Price_SMA,Price_SMA_ROC
0,EURUSD,2016-01-01,1.085800,1.085800,1.0858,0.000000,0.000000,0.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0858,NaN,1.085800,NaN
1,EURUSD,2016-01-04,1.084350,1.085413,1.0829,-0.000231,-0.000046,-0.000185,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0829,NaN,1.084350,NaN
2,EURUSD,2016-01-05,1.081100,1.083972,1.0746,-0.001072,-0.000251,-0.000821,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0746,NaN,1.081100,NaN
3,EURUSD,2016-01-06,1.080275,1.083149,1.0778,-0.001463,-0.000494,-0.000969,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0778,NaN,1.080275,NaN
4,EURUSD,2016-01-07,1.082900,1.084516,1.0934,-0.000509,-0.000497,-0.000012,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0934,NaN,1.082900,NaN


In [11]:
print("Kolone:", list(features.columns))
print("\nShape:", features.shape)


Kolone: ['Pair', 'Date', 'SMA', 'EMA', 'KAMA', 'MACD', 'MACD_signal', 'MACD_hist', 'ROC', 'ROC_SMA', 'SAR', 'SAR_SMA', 'CCI', 'CCI_SMA', 'RSI', 'RSI_SMA', 'CMO', 'CMO_SMA', 'RCI', 'RCI_SMA', 'STD', 'STD_SMA', 'SMA_ROC', 'EMA_ROC', 'KAMA_ROC', 'MACD_ROC', 'MACD_signal_ROC', 'MACD_hist_ROC', 'ROC_ROC', 'SAR_ROC', 'CCI_ROC', 'RSI_ROC', 'CMO_ROC', 'RCI_ROC', 'STD_ROC', 'Price', 'Price_ROC', 'Price_SMA', 'Price_SMA_ROC']

Shape: (13050, 39)


In [15]:
# Faza 2+3 - Features
with pd.ExcelWriter(CURRENCY_FEATURES_FILE, engine="openpyxl", datetime_format="YYYY-MM-DD") as writer:
    features.to_excel(writer, index=False)
print(f"Sačuvano: {CURRENCY_FEATURES_FILE}")


Sačuvano: Currency Data Features.xlsx


## Faza 5+6: Model neuronske mreže (MLP) + Eksperimenti

Pokreće 27 eksperimenata (1-1 do 3-3) za svaki valutni par.

**Napomena**: ovo može potrajati. Prvo testiraj sa `QUICK_TEST = True`
(samo 5 eksperimenata) da provereš da sve radi, pa onda pokreni puni set.


In [16]:
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, accuracy_score, f1_score,
                              precision_score, recall_score)

# ── KONFIGURACIJA ──────────────────────────────────────────────────────
N_PRICE_LAGS = 5
TEST_SIZE = 0.2

MLP_PARAMS = dict(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    max_iter=500,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42,
)

HORIZONS_1_1   = [1, 5, 10]
HORIZONS_OTHER = [1, 3, 5, 10]
EXP_3_3_AS_REGRESSION = False

# Test mode - postavi na True za brzu probu (5 eksperimenata)
QUICK_TEST = True
QUICK_TEST_EXPERIMENTS = ["1-1a", "1-1b", "1-1c", "2-1a", "3-1a"]

# Koliko parova obraditi (None = svi)
SERIES_TO_PROCESS = None
MAX_SERIES = 10

INDICATOR_BASE_COLS = ["SMA", "EMA", "KAMA", "MACD", "MACD_signal", "MACD_hist",
                       "ROC", "SAR", "CCI", "RSI", "CMO", "RCI", "STD"]
INDICATOR_SMA_COLS = ["ROC_SMA", "SAR_SMA", "CCI_SMA", "RSI_SMA", "CMO_SMA", "RCI_SMA", "STD_SMA"]
INDICATOR_ROC_COLS = [f"{c}_ROC" for c in INDICATOR_BASE_COLS]


def add_price_lags(df, n_lags=N_PRICE_LAGS):
    df = df.copy()
    for lag in range(1, n_lags + 1):
        df[f"Price_lag{lag}"] = df["Price"].shift(lag)
    return df


def add_targets(df, horizons):
    df = df.copy()
    for h in horizons:
        future_price = df["Price"].shift(-h)
        df[f"Target_Price_t{h}"] = future_price
        df[f"Target_Return_t{h}"] = (future_price - df["Price"]) / df["Price"] * 100
        df[f"Target_Sign_t{h}"] = (df[f"Target_Return_t{h}"] > 0).astype(int)
    return df


def get_feature_groups(df_columns):
    cols = set(df_columns)
    price_cols = ["Price"] + [f"Price_lag{i}" for i in range(1, N_PRICE_LAGS + 1)]
    return {
        "price": [c for c in price_cols if c in cols],
        "roc_price": [c for c in ["Price_ROC"] if c in cols],
        "sma_price": [c for c in ["Price_SMA"] if c in cols],
        "indicators": [c for c in INDICATOR_BASE_COLS if c in cols],
        "roc_indicators": [c for c in INDICATOR_ROC_COLS if c in cols],
        "sma_indicators": [c for c in INDICATOR_SMA_COLS if c in cols],
    }


def combine_groups(feature_groups, group_names):
    cols = []
    for g in group_names:
        for c in feature_groups.get(g, []):
            if c not in cols:
                cols.append(c)
    return cols


print("NN konfiguracija definisana.")


NN konfiguracija definisana.


In [17]:
def build_experiment_definitions():
    exps = {}
    exps["1-1a"] = dict(problem="regression", target_type="price", horizons=HORIZONS_1_1, input_groups=["price"])
    exps["1-1b"] = dict(problem="regression", target_type="price", horizons=HORIZONS_1_1, input_groups=["indicators"])
    exps["1-1c"] = dict(problem="regression", target_type="price", horizons=HORIZONS_1_1, input_groups=["price", "indicators"])

    exps["1-2a"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price"])
    exps["1-2b"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators"])
    exps["1-2c"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "indicators", "roc_price", "roc_indicators"])

    exps["1-3a"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
    exps["1-3b"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
    exps["1-3c"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])

    exps["2-1a"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price"])
    exps["2-1b"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["indicators"])
    exps["2-1c"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "indicators"])

    exps["2-2a"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price"])
    exps["2-2b"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators"])
    exps["2-2c"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "indicators", "roc_price", "roc_indicators"])

    exps["2-3a"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
    exps["2-3b"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
    exps["2-3c"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])

    exps["3-1a"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price"])
    exps["3-1b"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["indicators"])
    exps["3-1c"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "indicators"])

    exps["3-2a"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price"])
    exps["3-2b"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators"])
    exps["3-2c"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "indicators", "roc_price", "roc_indicators"])

    if EXP_3_3_AS_REGRESSION:
        exps["3-3a"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
        exps["3-3b"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
        exps["3-3c"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])
    else:
        exps["3-3a"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
        exps["3-3b"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
        exps["3-3c"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])

    return exps


experiments = build_experiment_definitions()

if QUICK_TEST:
    experiments = {k: v for k, v in experiments.items() if k in QUICK_TEST_EXPERIMENTS}
    print(f"[QUICK TEST] {len(experiments)} eksperimenata: {list(experiments.keys())}")
else:
    print(f"Ukupno eksperimenata: {len(experiments)}")
    for exp_id, cfg in experiments.items():
        print(f"  {exp_id}: {cfg['problem']:14s} | target={cfg['target_type']:6s} | horizonti={cfg['horizons']} | ulazi={cfg['input_groups']}")


[QUICK TEST] 5 eksperimenata: ['1-1a', '1-1b', '1-1c', '2-1a', '3-1a']


In [18]:
def run_mlp(df, feature_cols, target_col, problem, mlp_params=MLP_PARAMS, test_size=TEST_SIZE):
    data = df[feature_cols + [target_col]].dropna()
    if len(data) < 50:
        return None

    X = data[feature_cols].values
    y = data[target_col].values

    split = int(len(data) * (1 - test_size))
    if split < 20 or (len(data) - split) < 10:
        return None

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    if problem == "regression":
        model = MLPRegressor(**mlp_params)
        model.fit(X_train_s, y_train)
        preds = model.predict(X_test_s)
        return {
            "MAE": mean_absolute_error(y_test, preds),
            "RMSE": sqrt(mean_squared_error(y_test, preds)),
            "R2": r2_score(y_test, preds),
            "n_train": len(y_train),
            "n_test": len(y_test),
            "n_features": len(feature_cols),
        }
    else:
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            return None
        model = MLPClassifier(**mlp_params)
        model.fit(X_train_s, y_train)
        preds = model.predict(X_test_s)
        return {
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds, zero_division=0),
            "Recall": recall_score(y_test, preds, zero_division=0),
            "F1": f1_score(y_test, preds, zero_division=0),
            "n_train": len(y_train),
            "n_test": len(y_test),
            "n_features": len(feature_cols),
        }


def run_all_experiments_for_series(df_series, series_id, id_column, experiments):
    results = []
    df = df_series.sort_values("Date").reset_index(drop=True)
    df = add_price_lags(df, N_PRICE_LAGS)

    all_horizons = sorted(set(HORIZONS_1_1 + HORIZONS_OTHER))
    df = add_targets(df, all_horizons)

    feature_groups = get_feature_groups(df.columns)

    for exp_id, cfg in experiments.items():
        feature_cols = combine_groups(feature_groups, cfg["input_groups"])
        feature_cols = [c for c in feature_cols if c in df.columns]
        if not feature_cols:
            continue

        for h in cfg["horizons"]:
            target_type = cfg["target_type"]
            if target_type == "sign":
                target_col = f"Target_Sign_t{h}"
            elif target_type == "return":
                target_col = f"Target_Return_t{h}"
            else:
                target_col = f"Target_Price_t{h}"

            if target_col not in df.columns:
                continue

            metrics = run_mlp(df, feature_cols, target_col, cfg["problem"])
            if metrics is None:
                continue

            row = {
                id_column: series_id,
                "Experiment": exp_id,
                "Problem": cfg["problem"],
                "Target": target_col,
                "Horizon": h,
                "Input_groups": ",".join(cfg["input_groups"]),
            }
            row.update(metrics)
            results.append(row)

    return results


print("MLP funkcije definisane.")


MLP funkcije definisane.


In [20]:
if SERIES_TO_PROCESS is not None:
    series_ids = SERIES_TO_PROCESS
else:
    series_ids = features[ID_COLUMN].unique().tolist()[:MAX_SERIES]

print(f"Obrada {len(series_ids)} serija: {series_ids}")

all_results = []
for i, sid in enumerate(series_ids):
    print(f"\n[{i+1}/{len(series_ids)}] {sid}")
    df_series = features[features[ID_COLUMN] == sid]
    if len(df_series) < 50:
        print("    Premalo podataka, preskačem.")
        continue

    results = run_all_experiments_for_series(df_series, sid, ID_COLUMN, experiments)
    print(f"    Generisano {len(results)} rezultata.")
    all_results.extend(results)

results_df = pd.DataFrame(all_results)
print(f"\nUkupno {len(results_df)} redova rezultata.")
results_df.head()


,Pair,Experiment,Problem,Target,Horizon,Input_groups,MAE,RMSE,R2,n_train,n_test,n_features,Accuracy,Precision,Recall,F1
0,EURUSD,1-1a,regression,Target_Price_t1,1,price,0.005295,0.007120,0.974134,2083,521,6,NaN,NaN,NaN,NaN
1,EURUSD,1-1a,regression,Target_Price_t5,5,price,0.009868,0.012974,0.914266,2080,520,6,NaN,NaN,NaN,NaN
2,EURUSD,1-1a,regression,Target_Price_t10,10,price,0.012125,0.015799,0.873090,2076,519,6,NaN,NaN,NaN,NaN
3,EURUSD,1-1b,regression,Target_Price_t1,1,indicators,0.014859,0.019331,0.809997,2076,519,13,NaN,NaN,NaN,NaN
4,EURUSD,1-1b,regression,Target_Price_t5,5,indicators,0.019416,0.025559,0.667845,2072,519,13,NaN,NaN,NaN,NaN


In [21]:
reg = results_df[results_df["Problem"] == "regression"]
cls = results_df[results_df["Problem"] == "classification"]

if not reg.empty:
    print("Regresija - prosečne metrike po eksperimentu/horizontu:")
    display(reg.groupby(["Experiment", "Horizon"])[["MAE", "RMSE", "R2"]].mean().reset_index())

if not cls.empty:
    print("\nKlasifikacija - prosečne metrike po eksperimentu/horizontu:")
    display(cls.groupby(["Experiment", "Horizon"])[["Accuracy", "Precision", "Recall", "F1"]].mean().reset_index())


,Experiment,Horizon,Accuracy,Precision,Recall,F1
0,3-1a,1,0.505950,0.515577,0.551417,0.457584
1,3-1a,3,0.515547,0.630339,0.536178,0.453650
2,3-1a,5,0.513628,0.418729,0.621333,0.489800
3,3-1a,10,0.510940,0.415177,0.558454,0.462841


In [22]:
with pd.ExcelWriter(NN_RESULTS_FILE, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="All_Results", index=False)
    if not reg.empty:
        reg.to_excel(writer, sheet_name="Regression", index=False)
        reg.groupby(["Experiment", "Horizon"])[["MAE", "RMSE", "R2"]].mean().reset_index().to_excel(
            writer, sheet_name="Regression_Summary", index=False)
    if not cls.empty:
        cls.to_excel(writer, sheet_name="Classification", index=False)
        cls.groupby(["Experiment", "Horizon"])[["Accuracy", "Precision", "Recall", "F1"]].mean().reset_index().to_excel(
            writer, sheet_name="Classification_Summary", index=False)

print(f"Sačuvano: {NN_RESULTS_FILE}")


Sačuvano: NN_Experiment_Results.xlsx
